In [1]:
import os
import fcntl
import struct
from dataclasses import dataclass

### check device and permissions

In [ ]:
!ls -l /dev/mmcblk0

In [ ]:
!id

### raw block access (not needed at the end)

In [2]:
# Linux ioctl constants
BLKSSZGET = 0x1268      # logical sector size (u32)
BLKGETSIZE64 = 0x80081272  # device size in bytes (u64)

def get_sector_size(fd: int) -> int:
    buf = bytearray(4)
    fcntl.ioctl(fd, BLKSSZGET, buf, True)
    return struct.unpack("I", buf)[0]

def get_device_size(fd: int) -> int:
    buf = bytearray(8)
    fcntl.ioctl(fd, BLKGETSIZE64, buf, True)
    return struct.unpack("Q", buf)[0]

def read_blocks(dev: str, lba: int, count: int = 1) -> bytes:
    fd = os.open(dev, os.O_RDONLY)
    try:
        sector_size = get_sector_size(fd)
        dev_size = get_device_size(fd)

        offset = lba * sector_size
        length = count * sector_size

        if offset + length > dev_size:
            raise ValueError("Requested range exceeds device size")

        data = os.pread(fd, length, offset)
        if len(data) != length:
            raise IOError("Short read")
        return data
    finally:
        os.close(fd)


In [3]:
# Example: read first sector
dev = "/dev/mmcblk0"   # change to your device
data = read_blocks(dev, lba=0, count=1)
print(f"Read {len(data)} bytes")
print(data[:40].hex())

Read 512 bytes
02cc010000001600436172644c6f6767657220696e697469616c697a656402cc0100000004007069


### decode sequence of tag datasets

In [2]:
HEADER = struct.Struct("<HIH")  # u16, u32, u16

@dataclass
class DataBlock:
    type_label: int
    timestamp: int
    payload: bytes

In [3]:
def iter_blocks_stream(dev: str, start_offset: int = 0, chunk_size: int = 4096):
    with open(dev, "rb", buffering=0) as f:
        f.seek(start_offset)
        pending = b""
        stop = False

        while not stop:
            chunk = f.read(chunk_size)
            if not chunk:
                break

            pending += chunk
            offset = 0

            while offset + HEADER.size <= len(pending):
                type_label, timestamp, length = HEADER.unpack_from(pending, offset)

                if type_label == 0:
                    stop = True
                    break

                end = offset + HEADER.size + length
                if end > len(pending):
                    break

                payload = pending[offset + HEADER.size:end]
                yield DataBlock(type_label, timestamp, payload)
                offset = end

            pending = pending[offset:]

        if pending and not stop:
            raise ValueError("Trailing incomplete record at end of stream")

# Read and print logging data from card

In [4]:
log_data = [block for block in iter_blocks_stream("/dev/mmcblk0", start_offset=0)]
print(f"read {len(log_data)} blocks.")

read 212 blocks.


In [5]:
for block in log_data[:20]:
    if block.type_label == 0xcc02:
        msg = block.payload.decode("utf-8", errors="replace")
        print(f'{block.timestamp*0.001:08.3f} : {msg}')
    else:
        print(f'{block.timestamp*0.001:08.3f} : type {block.type_label:#04x}')

0000.001 : CardLogger initialized
0000.001 : ping
0000.102 : ping
0000.203 : ping
0000.304 : ping
0000.405 : ping
0000.506 : ping
0000.607 : ping
0000.708 : ping
0000.809 : ping
0000.910 : ping
0001.011 : ping
0001.112 : ping
0001.213 : ping
0001.314 : ping
0001.415 : ping
0001.516 : ping
0001.617 : ping
0001.718 : ping
0001.819 : ping
